# Imports

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from transformers import BertTokenizer, BertModel
from tqdm.auto import tqdm
from PIL import Image

In [3]:
# set device
device = 'mps'
print(f'Using device: {device}')

Using device: mps


# Constants (Paths, hyperparameters, etc.)

## Paths

In [4]:
# TASK = #'Suturing' # 'Needle_Passing' or 'Knot_Tying'

TASK = 'Suturing'

In [ ]:
META_FILE = f"../../datasets/jigsaws/{TASK}/meta_file_{TASK}.txt"
VIDEO_DIR = f"../../datasets/jigsaws/{TASK}/video/"      # JIGSAWS .avi files
LABEL_DIR = f"../../datasets/jigsaws/{TASK}/transcriptions/"      # CSVs with gesture annotations
OUTPUT_PATH = f"../../outputs/features/2025-11-16/jigsaws_full_processed_{TASK}.npz"
TRAIN_PATH = f"../../outputs/features/2025-11-16/jigsaws_train_{TASK}.npz"
TEST_PATH = f"../../outputs/features/2025-11-16/jigsaws_test_{TASK}.npz"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)


## Constants

### GRS (Global Rating Scale) Components

The JIGSAWS dataset includes 6 GRS components rated on a scale of 1-5:
- Respect for tissue : 0
- Suture/needle handling : 1
- Time and motion : 2
- Flow of operation : 3
- Overall performance : 4
- Quality of final product : 5

### Phase Mapping
- Phase 1 (Needle Preparation) - (G1, G5, G8, G12)
- Phase 2 (Needle Positioning) - (G2)
- Phase 3 (Needle Driving) - (G3)
- Phase 4 (Suture Management) - (G6, G7, G9, G15)
- Phase 5 (Transfer/Handoff) - (G4, G13, G14)
- Phase 6 (Completion) - (G10, G11)

- Gesture used in each task:
- Suturing:  1, 2, 3, 4, 5, 6, 8, 9, 10, 11
- Knot_Tying: 1 , 11 , 12 , 13 , 14 , 15
- NeedlePassing: 1, 2, 3, 4, 5, 6, 8, 11

In [ ]:
SUTURING_GESTURE_TO_PHASE = {
    'G1': 'P1', 'G5': 'P1', 'G8': 'P1',        # Phase 1: Needle Preparation
    'G2': 'P2',                                # Phase 2: Needle Positioning
    'G3': 'P3',                                # Phase 3: Needle Driving
    'G6': 'P4', 'G9': 'P4',                    # Phase 4: Suture Management
    'G4': 'P5',                                # Phase 5: Transfer / Handoff
    'G10': 'P6', 'G11': 'P6'                   # Phase 6: Completion / Final Positioning
}

KNOT_TYING_GESTURE_TO_PHASE = {
    'G1': 'P1', 'G12': 'P1',                   # Phase 1: Needle Preparation
    'G15': 'P4',                               # Phase 4: Suture Management
    'G13': 'P5', 'G14': 'P5',                  # Phase 5: Transfer / Handoff
    'G11': 'P6'                                # Phase 6: Completion / Final Positioning
}

NEEDLE_PASSING_GESTURE_TO_PHASE = {
    'G1': 'P1', 'G5': 'P1', 'G8': 'P1',        # Phase 1: Needle Preparation
    'G2': 'P2',                                # Phase 2: Needle Positioning
    'G3': 'P3',                                # Phase 3: Needle Driving
    'G6': 'P4',                                # Phase 4: Suture Management
    'G4': 'P5',                                # Phase 5: Transfer / Handoff
    'G11': 'P6'                                # Phase 6: Completion / Final Positioning
}

In [ ]:
#SURGEON_MAP = {"B": 0, "C": 1, "D": 2}  # adjust per dataset
SURGEON_MAP = {chr(c): i for i, c in enumerate(range(ord("B"), ord("J")))}  # B to I

print('Surgeon Map', SURGEON_MAP)

if TASK == "Suturing":
    LANG_TEXT = "suturing task"
elif TASK == "Needle_Passing":
    LANG_TEXT = "needle passing task"
else:
    LANG_TEXT = "knot tying task"

SEQ_LEN = 5
FRAME_SKIP = 5             # sample every 5th frame

if LANG_TEXT == "suturing task":
    GESTURE_TO_PHASE_TOKEN = SUTURING_GESTURE_TO_PHASE
elif LANG_TEXT == "knot tying task":
    GESTURE_TO_PHASE_TOKEN = KNOT_TYING_GESTURE_TO_PHASE
elif LANG_TEXT == "needle passing task":
    GESTURE_TO_PHASE_TOKEN = NEEDLE_PASSING_GESTURE_TO_PHASE

PHASE_VOCAB = ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']
PHASE_TOKEN_TO_ID = {p: i for i, p in enumerate(PHASE_VOCAB)}
PHASE_UNKNOWN_ID = len(PHASE_VOCAB)  # e.g., 6
print('Phase Token to ID:', PHASE_TOKEN_TO_ID)


# General Functions

In [ ]:
def get_language_embedding(text, tokenizer, model):
    tokens = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**tokens)
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0).numpy()
    return cls_embedding

def extract_frame_features(video_path, model, transform):
    cap = cv2.VideoCapture(video_path)
    features = []
    frame_indices = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_SKIP == 0:
            image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            x = transform(image).unsqueeze(0)
            x = x.to(device)
            with torch.no_grad():
                feat = model(x).squeeze().cpu().numpy()
            features.append(feat)
            frame_indices.append(frame_idx)
        frame_idx += 1
    cap.release()
    print(f"  📽️ Extracted {len(features)} frames from {video_path}")
    return np.stack(features), frame_indices

def align_gestures(label_path, frame_indices):
    gestures = np.full(len(frame_indices), 15, dtype=int) # 15 for mask
    phases = np.full(len(frame_indices), 6, dtype=int) # 6 for mask (P6 is unknown)
    ranges = []

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                start, end, g_label = int(parts[0]), int(parts[1]), parts[2]
                g_idx = int(g_label[1:]) - 1 # G1 -> 0
                ranges.append((start, end, g_idx))

    print("  Gesture label ranges (start-end-G):")
    for r in ranges[:5]:
        print("   ", r)

    for i, frame in enumerate(frame_indices):
        for start, end, g in ranges:
            if start <= frame <= end:
                gestures[i] = g
                g_name = f'G{g+1}'
                phase_token = GESTURE_TO_PHASE_TOKEN.get(g_name, None)
                phases[i] = PHASE_TOKEN_TO_ID.get(phase_token, PHASE_UNKNOWN_ID)
                break
    print(f"  First 20 frame indices: {frame_indices[:20]}")
    print(f"  First 20 gesture labels: {gestures[:20]}")
    print(f"  First 20 phase labels: {phases[:20]}")
    
    return gestures, phases

def process_trial(video_file, lang_embed, vision_model, grs, transform):
    trial_name = os.path.splitext(video_file)[0]
    
    if LANG_TEXT == "suturing task":
        parts = trial_name.split("_")
        if len(parts) < 2:
            print(f"Skipping malformed filename: {trial_name}")
            return []

        surgeon_code = parts[1][0]
        surgeon_id = SURGEON_MAP.get(surgeon_code, -1)
        print(surgeon_id)
        if surgeon_id == -1:
            print(f"Unknown surgeon code in {video_file}")
            return []

        video_path = os.path.join(VIDEO_DIR, video_file)
        # Extract "Suturing_B001" from "Suturing_B001_capture1"
        base_trial_id = "_".join(trial_name.split("_")[:2])
    else:
        parts = trial_name.split("_")
        if len(parts) < 3:
            print(f"Skipping malformed filename: {trial_name}")
            return []

        surgeon_code = parts[2][0]
        surgeon_id = SURGEON_MAP.get(surgeon_code, -1)
        print(surgeon_id)
        if surgeon_id == -1:
            print(f"Unknown surgeon code in {video_file}")
            return []

        video_path = os.path.join(VIDEO_DIR, video_file)
        # Extract "Needle_Passing_B001" from "Needle_Passing_B001_capture1"
        base_trial_id = "_".join(trial_name.split("_")[:3])


    label_path = os.path.join(LABEL_DIR, base_trial_id + ".txt")
    print(f"\n🔍 Processing video: {video_file}")
    print(f"→ Using label file: {label_path}")


    if not os.path.exists(label_path):
        print(f"Missing label file for {trial_name}")
        return []

    vision, frame_indices = extract_frame_features(video_path, vision_model, transform)
    gestures, phases = align_gestures(label_path, frame_indices)

    samples = []
    for i in range(0, len(vision) - SEQ_LEN + 1, SEQ_LEN):
        samples.append({
            "vision_feat": vision[i:i+SEQ_LEN].mean(axis=0),
            "lang_feat": lang_embed,
            "gesture_seq": gestures[i:i+SEQ_LEN],
            "phase_seq": phases[i:i+SEQ_LEN],
            "surgeon_id": surgeon_id,
            'grs_components' : grs.get(base_trial_id)
        })
    print(f"  ✅ Appended {len(samples)} sequences for {video_file}")
    print(f"    First gesture seq: {samples[0]['gesture_seq']}" if samples else "    No valid samples.")
    return samples
    




In [ ]:
def get_grs_from_meta_file(meta=META_FILE):
    print(f"Opening meta file: {meta}")
    with open(meta, "r") as f:
        lines = [line.strip().replace("\t\t","\t").split("\t") for line in f if line.strip()]
        print('sample line: ', lines[0])
    print(f"Found {len(lines)}")

    grs = {}

    for line in lines:
        video_name = line[0]
        # Store all 6 individual GRS components (7th is mask)
        grs_components = [int(num) - 1 for num in line[3:9]]  # 6 individual scores (indices 3-8) (-1 for zero indexing)
        grs[video_name] = np.array(grs_components, dtype=np.float32)  # 6 individual scores

    print('grs example:', dict(list(grs.items())[:1]))
    return grs

grs = get_grs_from_meta_file()

# Main

## Load Models

In [ ]:
# Vision Model
vision_model = resnet18(weights=ResNet18_Weights.DEFAULT)
vision_model.eval().to(device)

# Language Model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
language_model = BertModel.from_pretrained('bert-base-uncased')
language_model.eval()

print("Models loaded.")

## Transforms

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

## Main

In [ ]:
# Get language embedding
lang_embed = get_language_embedding(LANG_TEXT, tokenizer, language_model)

# Process videos to extract features and labels(gestures)
all_samples = []

for video_file in tqdm(sorted(os.listdir(VIDEO_DIR))):
    if not video_file.endswith(".avi"):
        continue
    trial_samples = process_trial(video_file, lang_embed, vision_model, grs, transform)
    all_samples.extend(trial_samples)

vision_feats = np.stack([s["vision_feat"] for s in all_samples])
lang_feats = np.stack([s["lang_feat"] for s in all_samples])
gesture_seqs = np.stack([s["gesture_seq"] for s in all_samples])
phase_seqs = np.stack([s["phase_seq"] for s in all_samples])
surgeon_ids = np.array([s["surgeon_id"] for s in all_samples])
grs_components = np.stack([s["grs_components"] for s in all_samples])

np.savez_compressed(
    OUTPUT_PATH,
    vision_feat = vision_feats,
    lang_feat = lang_feats,
    gesture_seq = gesture_seqs,
    phase_seq = phase_seqs,
    surgeon_id = surgeon_ids,
    grs_components = grs_components             
)


print(f"Total samples: {len(all_samples)}")
print(f"Vision features shape: {vision_feats.shape}")
print(f"Language features shape: {lang_feats.shape}")

In [ ]:
# split into train and test sets (80:20)
data = np.load(OUTPUT_PATH)
data_size = len(data['vision_feat'])
train_size = int(0.8 * data_size)
test_size = data_size - train_size

print('data size:', data_size)
print('train size:', train_size)
print('test size:', test_size)

# Shuffle indices
indices = np.arange(data_size)

np.random.shuffle(indices)

train_indices = indices[:train_size]
test_indices = indices[train_size:]

train_set = {key: data[key][train_indices] for key in data.keys()}
test_set = {key: data[key][test_indices] for key in data.keys()}

np.savez_compressed(TRAIN_PATH, **train_set)
np.savez_compressed(TEST_PATH, **test_set)

# Verify Data

In [ ]:
import numpy as np
import random

data = np.load(OUTPUT_PATH)
print("Keys in file:", data.files)

numbers = sorted([random.randint(0, 4000) for _ in range(10)])

for i in numbers:  # random 10 samples
    print(f"Sample {i}:")
    print("  gesture_seq:", data["gesture_seq"][i])
    print("  phase_seq:", data["phase_seq"][i])
    print("  surgeon_id:", data["surgeon_id"][i])
    print("  vision_feat[:5]:", data["vision_feat"][i][:5])
    print("  lang_feat[:5]:", data["lang_feat"][i][:5])
    print("  grs_components:", data["grs_components"][i])


In [ ]:
print(data['grs_components'].shape)
print(np.unique(data['grs_components']))
print(np.unique(data['surgeon_id']))
print(np.unique(data['gesture_seq']))
print(np.unique(data['phase_seq']))


In [ ]:
# surgeon id to grs component mapping (grs 1-5 -> 0-4)
for surgeon_id in np.unique(data['surgeon_id']):
    indices = np.where(data['surgeon_id'] == surgeon_id)[0]
    grs_for_surgeon = np.unique(data['grs_components'][indices], axis=0)
    print(f"Surgeon ID {surgeon_id} GRS components:\n", grs_for_surgeon)

In [ ]:
# data balance check for each grs component
for j in range(6):
    unique_scores = np.unique(data['grs_components'][:, j])
    print(f"\nGRS Component {j+1}:")
    print(f"  Unique scores: {unique_scores + 1}")
    total_count = 0
    for score in unique_scores:
        count = np.sum(data['grs_components'][:, j] == score)
        print(f"    Score {score + 1}: Count = {count}")
        total_count += count
    print(f"  Total: {total_count}")